# 08 — The Payoff: Multi-Agent with the Official SDK

## Why this notebook exists

For seven notebooks we hand-rolled every byte of A2A. JSON-RPC envelopes, task state machines, SSE frames, webhook signatures, OAuth flows. The goal was making the protocol *legible*: if you've stuck with the series, you can now look at any A2A interaction in the wild and know what's happening.

**You don't want to keep writing that code.** In production you reach for an SDK that handles the protocol mechanics so your code can focus on the agent's behavior.

This notebook installs Google's official **`a2a-sdk`**, rebuilds the researcher and writer from notebook 01 using the SDK's `AgentExecutor` pattern, and orchestrates them with an SDK-driven coordinator. It closes with a side-by-side diff that shows how much hand-rolling the SDK absorbs.

> *Pins `a2a-sdk==0.3.26` (the v0.3 compatibility line). The 1.x line targets the A2A 1.0 spec, which has wire-format differences from what we've been building.*

## What you'll learn

- How the `a2a-sdk` decomposes an A2A server: `AgentExecutor` (your code), `DefaultRequestHandler` (protocol glue), `InMemoryTaskStore` (state), and an `A2AStarletteApplication` that wires the routes into Starlette.
- How to build a typed `AgentCard` programmatically with the SDK's classes.
- How an `AgentExecutor` enqueues `TaskStatusUpdateEvent` and `TaskArtifactUpdateEvent` instances — the same events you built by hand in notebook 05, just constructed via the SDK's types (or its `TaskUpdater` helper).
- How an A2A *client* (the coordinator) uses `A2ACardResolver` to discover an agent and then sends typed `SendMessageRequest` envelopes.
- Why the side-by-side line count matters: it's the cost of *not* having a standard.
- Where to go next: real LLMs behind the agents, multi-language interop, production deployment.

## 1. Setup

Install `a2a-sdk` (pinned to the v0.3-compatible line) and import the helpers we'll use throughout.

In [ ]:
import sys
import subprocess

# The Starlette + SSE wiring lives in the `[http-server]` extras, so install both
# the SDK and that extras bundle to get a server we can stand up in-notebook.
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "a2a-sdk[http-server]==0.3.26"])
print("a2a-sdk installed")

In [ ]:
import asyncio
import threading
import time
import uuid

import httpx
import uvicorn

# SDK server surfaces. In a2a-sdk 0.3.26 the symbols live where you'd expect,
# though some names differ from earlier plan drafts (notably no separate
# `create_*_routes` helpers — the Starlette app builder owns the routes).
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore, TaskUpdater
from a2a.server.apps import A2AStarletteApplication

# Typed wire models — same shape as the JSON we hand-rolled, just as pydantic
# classes. Note `AgentInterface(transport=..., url=...)` and that `TaskState`
# is a plain string enum (TaskState.working, .completed, .failed), not a
# protobuf-wrapped value.
from a2a.types import (
    AgentCapabilities,
    AgentCard,
    AgentInterface,
    AgentSkill,
    Message,
    MessageSendParams,
    Part,
    Role,
    SendMessageRequest,
    Task,
    TaskArtifactUpdateEvent,
    TaskState,
    TaskStatus,
    TaskStatusUpdateEvent,
    TextPart,
)

# SDK helpers for building messages, artifacts, and fresh tasks from a user
# message — the building blocks our hand-rolled code spelled out by hand.
from a2a.utils import (
    new_agent_text_message,
    new_task,
    new_text_artifact,
)

# Client surfaces.
from a2a.client import A2ACardResolver, A2AClient

# Background-thread uvicorn helper (same as previous notebooks).
_servers: list[uvicorn.Server] = []


def run_server_in_thread(app, port: int) -> uvicorn.Server:
    config = uvicorn.Config(app, host="127.0.0.1", port=port, log_level="warning")
    server = uvicorn.Server(config)
    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()
    for _ in range(50):
        if server.started:
            break
        time.sleep(0.05)
    else:
        raise RuntimeError(f"Server on port {port} did not start in time")
    _servers.append(server)
    return server


def shutdown_all_servers() -> None:
    for server in list(_servers):
        server.should_exit = True
    _servers.clear()


print("Setup OK")

## 2. A Tour of the SDK

| SDK piece | What it replaces (vs. our hand-rolled code) |
|---|---|
| `AgentExecutor` (subclass it; implement `async execute(ctx, queue)`) | The whole `handle_jsonrpc` function from notebook 03 — method dispatch, task creation, status updates, artifact assembly. |
| `RequestContext` | Parsed `Message`, `task_id`, `context_id` — instead of plucking them out of `req.params["message"]` by hand. `context.get_user_input()` even joins all text parts for you. |
| `EventQueue` | Replaces the `_set_status` / `_set_artifacts` lock-protected stores plus the SSE generator — you `await event_queue.enqueue_event(thing)` and the SDK handles delivery for both `message/send` (last event wins) and `message/stream` (streamed). |
| `TaskUpdater` (helper around `EventQueue`) | The bookkeeping for "transition this task to working / add this artifact / mark complete" — fewer fields to set by hand. |
| `DefaultRequestHandler` | The router that maps JSON-RPC method names to `AgentExecutor` callbacks plus task lifecycle plumbing (the entire notebook 04 state machine). |
| `InMemoryTaskStore` | Our `TASKS` dict + `STORE_LOCK` from notebook 04. |
| `A2AStarletteApplication(...).build()` | The FastAPI route declarations from notebooks 02 and 03 — mounts the agent-card endpoint at `/.well-known/agent-card.json` and the JSON-RPC endpoint at `/`. |
| `AgentCard`, `AgentCapabilities`, `AgentSkill`, `AgentInterface` (types) | Our hand-written pydantic `AgentCard` model from notebook 02. |
| `new_task`, `new_text_artifact`, `new_agent_text_message` (helpers) | Boilerplate task/artifact/message construction. |
| `TaskStatusUpdateEvent`, `TaskArtifactUpdateEvent` (types) | Our hand-written event models from notebook 05. |
| `A2ACardResolver`, `A2AClient` (client side) | The hand-rolled JSON-RPC envelope construction + `httpx.post` calls. |

Same protocol on the wire — what the SDK does is absorb the *mechanics* of producing and consuming it. Field names in Python switch from `defaultInputModes` (the JSON wire shape we've been using) to `default_input_modes` (the SDK's snake_case Python convention); the SDK serializes back to the spec-correct camelCase on the wire.